In [1]:

import pandas as pd
df = pd.read_csv(r'D:\Github\ML projects\Telco-churn-classification\data\final_data.csv')
df.head()

,const,gender,SeniorCitizen,Partner,Dependents,tenure,PaperlessBilling,MultipleLines_Yes,InternetService_Fiber optic,OnlineSecurity_Yes,...,TechSupport_Yes,StreamingTV_Yes,Contract_One year,Contract_Two year,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,PC1,PC2,Churn
0,1.0,0,0,1,0,1,1,0,0,0,...,0,0,0,0,0,1,0,-1.923711,-4.109683,0
1,1.0,1,0,0,0,34,0,0,0,1,...,0,0,1,0,0,0,1,-1.322171,0.634423,0
2,1.0,1,0,0,0,2,1,0,0,1,...,0,0,0,0,0,0,1,-1.322171,0.634423,1
3,1.0,1,0,0,0,45,0,0,0,1,...,1,0,1,0,0,0,0,-1.923711,-4.109683,0
4,1.0,0,0,0,0,2,1,0,1,0,...,0,0,0,0,0,1,0,-1.322171,0.634423,1


In [2]:
df['Churn'].value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
from sklearn.metrics import classification_report
import pandas as pd
import time
X = df.drop('Churn', axis=1)
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
proba = rf.predict_proba(X_test)[:, 1]
y_pred = rf.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.83      0.90      0.87      1036
           1       0.64      0.51      0.57       373

    accuracy                           0.79      1409
   macro avg       0.74      0.70      0.72      1409
weighted avg       0.78      0.79      0.79      1409



In [5]:
from sklearn.metrics import precision_score, recall_score, f1_score
proba = rf.predict_proba(X_test)[:, 1]
print("threshold tuning for Random Forest")
print(f"{'Thresh':<8}{'Prec_1':<8}{'Rec_1':<8}{'F1_1':<8}")
for thresh in [0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
    preds = (proba >= thresh).astype(int)
    prec = precision_score(y_test, preds, pos_label=1)
    rec = recall_score(y_test, preds, pos_label=1)
    f1 = f1_score(y_test, preds, pos_label=1)
    print(f"{thresh:<8}{prec:<8.3f}{rec:<8.3f}{f1:<8.3f}")

threshold tuning for Random Forest
Thresh  Prec_1  Rec_1   F1_1    
0.25    0.490   0.828   0.616   
0.3     0.522   0.753   0.617   
0.35    0.555   0.676   0.609   
0.4     0.589   0.611   0.600   
0.45    0.606   0.560   0.582   
0.5     0.632   0.507   0.562   


In [6]:
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
import time

lgbm = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# Training timer
start_train = time.time()
lgbm.fit(X_train, y_train)
train_time = time.time() - start_train
print(f"⏱ Training time: {train_time:.2f} seconds")

# Prediction timer
start_pred = time.time()
proba = lgbm.predict_proba(X_test)[:, 1]
y_pred = (proba >= 0.5).astype(int)
pred_time = time.time() - start_pred
print(f"⏱ Prediction time: {pred_time:.4f} seconds")

# Classification report
print(classification_report(y_test, y_pred, digits=3))
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.3f}")

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 1496, number of negative: 4138
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000900 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 115
[LightGBM] [Info] Number of data points in the train set: 5634, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
⏱ Training time: 1.60 seconds
⏱ Prediction time: 0.0118 seconds
              precision    recall  f1-score   support

           0      0.892     0.772     0.828      1036
           1      0.539     0.740     0.624       373

    accuracy                          0.764      1409
   macro avg      0.715     0.756     0.726      1409
weighted avg      0.798     0.

In [7]:
from sklearn.metrics import precision_score, recall_score, f1_score

proba = lgbm.predict_proba(X_test)[:, 1]

print("Threshold tuning for LightGBM")

print(f"{'Thresh':<8}{'Prec_1':<8}{'Rec_1':<8}{'F1_1':<8}")
for thresh in [0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
    preds = (proba >= thresh).astype(int)
    prec = precision_score(y_test, preds, pos_label=1)
    rec = recall_score(y_test, preds, pos_label=1)
    f1 = f1_score(y_test, preds, pos_label=1)
    print(f"{thresh:<8}{prec:<8.3f}{rec:<8.3f}{f1:<8.3f}")

Threshold tuning for LightGBM
Thresh  Prec_1  Rec_1   F1_1    
0.25    0.459   0.877   0.603   
0.3     0.479   0.858   0.615   
0.35    0.494   0.834   0.620   
0.4     0.505   0.815   0.624   
0.45    0.530   0.786   0.633   
0.5     0.539   0.740   0.624   


In [8]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
import time

# Calculate scale_pos_weight for imbalance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss'
)

# Training timer
start_train = time.time()
xgb.fit(X_train, y_train)
train_time = time.time() - start_train
print(f"⏱ Training time: {train_time:.2f} seconds")

# Prediction timer
start_pred = time.time()
proba = xgb.predict_proba(X_test)[:, 1]
y_pred = (proba >= 0.6).astype(int)
pred_time = time.time() - start_pred
print(f"⏱ Prediction time: {pred_time:.4f} seconds")

# Classification report
print(classification_report(y_test, y_pred, digits=3))
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.3f}")

⏱ Training time: 1.14 seconds
⏱ Prediction time: 0.0160 seconds
              precision    recall  f1-score   support

           0      0.869     0.837     0.853      1036
           1      0.589     0.649     0.617       373

    accuracy                          0.787      1409
   macro avg      0.729     0.743     0.735      1409
weighted avg      0.795     0.787     0.790      1409

Accuracy: 0.787


In [9]:
from sklearn.metrics import precision_score, recall_score, f1_score

proba = xgb.predict_proba(X_test)[:, 1]

print("Threshold tuning for XGBoost")

print(f"{'Thresh':<8}{'Prec_1':<8}{'Rec_1':<8}{'F1_1':<8}")
for thresh in [0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
    preds = (proba >= thresh).astype(int)
    prec = precision_score(y_test, preds, pos_label=1)
    rec = recall_score(y_test, preds, pos_label=1)
    f1 = f1_score(y_test, preds, pos_label=1)
    print(f"{thresh:<8}{prec:<8.3f}{rec:<8.3f}{f1:<8.3f}")

Threshold tuning for XGBoost
Thresh  Prec_1  Rec_1   F1_1    
0.25    0.460   0.863   0.600   
0.3     0.481   0.845   0.613   
0.35    0.489   0.810   0.609   
0.4     0.509   0.791   0.619   
0.45    0.533   0.764   0.628   
0.5     0.549   0.745   0.633   


In [10]:
import optuna
from xgboost import XGBClassifier
from sklearn.metrics import recall_score
from sklearn.model_selection import train_test_split

# Objective function for Optuna
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 800),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 0, 5),
        "reg_lambda": trial.suggest_float("reg_lambda", 0, 5),
        "random_state": 42,
        "n_jobs": -1,
        "scale_pos_weight": (y_train == 0).sum() / (y_train == 1).sum(),
        "eval_metric": "logloss"
    }
    
    model = XGBClassifier(**params)
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    y_pred = (proba >= 0.3).astype(int)  # Keep your tuned threshold
    return recall_score(y_test, y_pred, pos_label=1)  # Optimize recall for churners

# Run Optuna
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

print("Best Params:", study.best_params)
print("Best Recall:", study.best_value)

c:\Users\ankus\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-07-01 18:37:47,289] A new study created in memory with name: no-name-4d0e342f-1ec4-4a8d-9577-5b87abdadfb4
[I 2026-07-01 18:37:47,922] Trial 0 finished with value: 0.9276139410187667 and parameters: {'n_estimators': 453, 'learning_rate': 0.13811027837012144, 'max_depth': 4, 'subsample': 0.628843641454712, 'colsample_bytree': 0.6742258991381425, 'min_child_weight': 9, 'gamma': 2.298751658808715, 'reg_alpha': 3.058220353973585, 'reg_lambda': 4.81686121123719}. Best is trial 0 with value: 0.9276139410187667.
[I 2026-07-01 18:37:48,249] Trial 1 finished with value: 0.9195710455764075 and parameters: {'n_estimators': 328, 'learning_rate': 0.13482239536751844, 'max_depth': 4, 'subsample': 0.5378326796787347, 'co

Best Params: {'n_estimators': 378, 'learning_rate': 0.012007807383198631, 'max_depth': 3, 'subsample': 0.7327451610323206, 'colsample_bytree': 0.8686716717736397, 'min_child_weight': 8, 'gamma': 4.4303893886109265, 'reg_alpha': 2.560030103959993, 'reg_lambda': 3.4135940832240035}
Best Recall: 0.9463806970509383


In [11]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import time

# Calculate scale_pos_weight for imbalance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# Add the scale_pos_weight and fixed params to the best ones from Optuna
best_params = study.best_params
best_params.update({
    "random_state": 42,
    "n_jobs": -1,
    "scale_pos_weight": scale_pos_weight,
    "eval_metric": "logloss"
})

# Create model from best params
xgb = XGBClassifier(**best_params)

# Training timer
start_train = time.time()
xgb.fit(X_train, y_train)
train_time = time.time() - start_train
print(f"⏱ Training time: {train_time:.2f} seconds")

# Prediction timer
start_pred = time.time()
proba = xgb.predict_proba(X_test)[:, 1]
y_pred = (proba >= 0.3).astype(int)
pred_time = time.time() - start_pred
print(f"⏱ Prediction time: {pred_time:.4f} seconds")

# Classification report
print(classification_report(y_test, y_pred, digits=3))

⏱ Training time: 0.36 seconds
⏱ Prediction time: 0.0078 seconds
              precision    recall  f1-score   support

           0      0.965     0.536     0.689      1036
           1      0.423     0.946     0.585       373

    accuracy                          0.644      1409
   macro avg      0.694     0.741     0.637      1409
weighted avg      0.822     0.644     0.661      1409

